# Fire Season Timing - Turkey Ecoregion Scale

In [10]:
'''
Computes fire season timing metrics (onset, peak, end, season length)
for all WWF RESOLVE ecoregions intersecting Turkey, for years 2003-2025.

Data sources:
- MODIS Terra active fire: MODIS/061/MOD14A1
- MODIS Aqua active fire:  MODIS/061/MYD14A1
- Ecoregions:              RESOLVE/ECOREGIONS/2017
- Country boundary:        USDOS/LSIB_SIMPLE/2017

Output:
- outputs/turkey_ecoregions/<ECO_ID>_<ECO_NAME>.csv  (per ecoregion)
- outputs/turkey_ecoregions/master_turkey.csv        (combined)
'''

import ee
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import calendar
import datetime
from tqdm import tqdm

In [11]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [12]:
# PARAMETERS ---------------------------------------------------------------------------------------

FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2026))  # Full study period: 2003–2025

In [13]:
# LOAD MODIS COLLECTIONS ---------------------------------------------------------------------------
# Terra and Aqua are loaded once here at module level.
# Per-year and per-day filtering is handled inside get_daily_counts().

terra = ee.ImageCollection("MODIS/061/MOD14A1").select('FireMask')
aqua  = ee.ImageCollection("MODIS/061/MYD14A1").select('FireMask')

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())
print('Terra and Aqua collections loaded.')

Terra image count: 9439
Aqua image count: 8627
Terra and Aqua collections loaded.


In [14]:
# LOAD TURKEY ECOREGIONS ---------------------------------------------------------------------------

# Turkey boundary from LSIB
turkey = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'Turkey'))

# RESOLVE ecoregions clipped to Turkey
ecoregions_turkey = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(turkey.geometry())

# Inspect
n_eco    = ecoregions_turkey.size().getInfo()
eco_list = ecoregions_turkey.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']).getInfo()

print(f'Number of ecoregions intersecting Turkey: {n_eco}')
print()
for f in eco_list['features']:
    p = f['properties']
    print(p['ECO_ID'], '|', p['ECO_NAME'], '|', p['BIOME_NAME'])

Number of ecoregions intersecting Turkey: 14

786 | Anatolian conifer and deciduous mixed forests | Mediterranean Forests, Woodlands & Scrub
804 | Southern Anatolian montane conifer and deciduous forests | Mediterranean Forests, Woodlands & Scrub
646 | Balkan mixed forests | Temperate Broadleaf & Mixed Forests
650 | Caucasus mixed forests | Temperate Broadleaf & Mixed Forests
665 | Euxine-Colchic broadleaf forests | Temperate Broadleaf & Mixed Forests
652 | Central Anatolian steppe and woodlands | Temperate Broadleaf & Mixed Forests
662 | Eastern Anatolian deciduous forests | Temperate Broadleaf & Mixed Forests
688 | Zagros Mountains forest steppe | Temperate Broadleaf & Mixed Forests
703 | Northern Anatolian conifer and deciduous forests | Temperate Conifer Forests
725 | Central Anatolian steppe | Temperate Grasslands, Savannas & Shrublands
727 | Eastern Anatolian montane steppe | Temperate Grasslands, Savannas & Shrublands
739 | Syrian xeric grasslands and shrublands | Temperate Gras

In [15]:
eco_records = []
for f in eco_list['features']:
    p            = f['properties']
    full_geom    = ee.Geometry(f['geometry'])
    clipped_geom = full_geom.intersection(turkey.geometry(), maxError=100)

    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : clipped_geom
    })

print(f'Built {len(eco_records)} clipped ecoregion records.')

Built 14 clipped ecoregion records.


In [16]:
import json
from shapely.geometry import mapping

# Save clipped geometries to GeoJSON for reuse in other notebooks
geo_records_export = []
for rec in eco_records:
    geo_records_export.append({
        'eco_id'  : rec['eco_id'],
        'eco_name': rec['eco_name'],
        'geometry': rec['geometry'].getInfo()   # fetch clipped geometry from GEE once
    })

os.makedirs('outputs/turkey_ecoregions', exist_ok=True)

with open('outputs/turkey_ecoregions/eco_geometries.json', 'w') as f:
    json.dump(geo_records_export, f)

print(f'Saved {len(geo_records_export)} clipped geometries.')

Saved 14 clipped geometries.


## Helper Functions

In [17]:
# FUNCTION: get_daily_counts -----------------------------------------------------------------------


def get_daily_counts(eco_geometry, year):
    """
    Compute daily MODIS active fire detection counts for a given
    ecoregion geometry and calendar year.

    Combines Terra (MOD14A1) and Aqua (MYD14A1) by taking the pixel-wise
    maximum across sensors for each day, deduplicating detections that
    appear in both sensors on the same day.

    All 365 daily counts are retrieved in a SINGLE reduceRegion call
    by stacking all daily images into one multi-band image using toBands().
    This avoids the 'Too many concurrent aggregations' error that occurs
    when reduceRegion is called inside a mapped function.

    Parameters
    ----------
    eco_geometry : ee.Geometry
        The geometry of the ecoregion to compute counts for.
    year : int
        The calendar year to process (e.g. 2008).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
          - doy           : int, day of year (1-indexed)
          - n_detections  : int, number of fire pixels detected
        One row per day of the year (365 or 366 rows).
    """

    start  = ee.Date.fromYMD(year, 1, 1)
    end    = ee.Date.fromYMD(year + 1, 1, 1)
    n_days = 366 if calendar.isleap(year) else 365

    # Pre-filter both collections to this year
    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)

    # Fallback empty image for days where a sensor returns no image
    empty = ee.Image.constant(0).rename('FireMask').toUint8()

    # Server-side list of day offsets: [0, 1, 2, ... n_days-1]
    day_seq = ee.List.sequence(0, n_days - 1)

    def make_daily_image(d):
        """
        For a single day offset d, build a deduplicated binary fire image.
        Returns a single-band image named by its DOY (e.g. 'day_001').
        No reduceRegion here — reduction happens once outside this function.
        """
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        terra_day = terra_year.filterDate(date, date_end)
        aqua_day  = aqua_year.filterDate(date, date_end)

        # Use empty fallback if sensor has no image for this day
        t = ee.Image(ee.Algorithms.If(
            terra_day.size().gt(0),
            terra_day.select('FireMask').max(),
            empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_day.size().gt(0),
            aqua_day.select('FireMask').max(),
            empty
        ))

        # Pixel-wise max across sensors = deduplication
        combined    = t.max(a)
        fire_binary = combined.gte(FIRE_MASK_MIN).unmask(0)

        # Name this band by its DOY so we can identify it after toBands()
        # ee.Number.format creates a string like '001', '002', ... '365'
        band_name = ee.String('day_').cat(
            d.add(1).toInt().format('%03d')
        )

        return fire_binary.rename(band_name)

    # Build a collection of 365 single-band images
    daily_collection = ee.ImageCollection(day_seq.map(make_daily_image))

    # Stack all 365 bands into ONE multi-band image
    # This is the key change — instead of 365 separate images,
    # we now have one image with 365 bands (one per day)
    stacked = daily_collection.toBands()

    # ONE single reduceRegion call on the entire stacked image
    # This counts fire pixels for all 365 days in a single aggregation
    counts_dict = stacked.reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = eco_geometry,
        scale     = 1000,
        maxPixels = 1e9
    ).getInfo()  # one single getInfo() call — brings all 365 counts at once

    # counts_dict looks like: {'0_day_001': 5, '1_day_002': 0, ...}
    # Parse it back into a tidy DataFrame
    rows = []
    for band_name, count in sorted(counts_dict.items()):
        # Extract the DOY number from the band name (e.g. '0_day_001' → 1)
        doy = int(band_name[-3:])
        rows.append({
            'doy'         : doy,
            'n_detections': int(count) if count is not None else 0
        })

    return pd.DataFrame(rows)

In [18]:
# FUNCTION: compute_timing_metrics -----------------------------------------------------------------

def compute_timing_metrics(df, year):
    """
    Onset and end: 5%/95% cumulative detection thresholds.
    Peak: fire activity centroid (detection-weighted mean DOY).
    Added fields: onset_month, peak_month, peak_outside_window.
    Returns None if total detections < MIN_DETECTIONS.
    """

    total = df['n_detections'].sum()

    if total < MIN_DETECTIONS:
        print(f'  {year}: insufficient detections ({total}), skipping.')
        return None

    df = df.copy().sort_values('doy')
    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    onset_rows = df[cum_frac >= ONSET_THRESHOLD]
    end_rows   = df[cum_frac >= END_THRESHOLD]

    if onset_rows.empty or end_rows.empty:
        print(f'  {year}: could not compute onset or end, skipping.')
        return None

    onset_doy = int(onset_rows.iloc[0]['doy'])
    end_doy   = int(end_rows.iloc[0]['doy'])

    weights  = df['n_detections']
    peak_doy = int(round((df['doy'] * weights).sum() / weights.sum()))

    peak_outside_window = not (onset_doy <= peak_doy <= end_doy)
    if peak_outside_window:
        print(f'  {year}: WARNING — peak ({peak_doy}) outside onset-end window '
              f'({onset_doy}-{end_doy}), flagging.')

    season_length = end_doy - onset_doy + 1

    onset_month = (datetime.date(year, 1, 1) + datetime.timedelta(days=onset_doy - 1)).month
    peak_month  = (datetime.date(year, 1, 1) + datetime.timedelta(days=peak_doy  - 1)).month

    return {
        'year'               : year,
        'onset_doy'          : onset_doy,
        'peak_doy'           : peak_doy,
        'end_doy'            : end_doy,
        'season_length'      : season_length,
        'n_detections'       : int(total),
        'onset_month'        : onset_month,
        'peak_month'         : peak_month,
        'peak_outside_window': int(peak_outside_window)
    }

## Main Pipeline

In [19]:
# FULL PIPELINE LOOP - ALL ECOREGIONS x ALL YEARS --------------------------------------------------

from tqdm import tqdm

output_dir = r'C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions'
daily_dir  = r'C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\daily_counts'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(daily_dir,  exist_ok=True)

all_metrics  = []
failed_years = []

for eco in tqdm(eco_records, desc='Ecoregions'):
    eco_id    = eco['eco_id']
    eco_name  = eco['eco_name']
    biome_num = eco['biome_num']
    biome_name= eco['biome_name']
    geometry  = eco['geometry']

    safe_name  = eco_name.replace(' ', '_').replace('/', '_')
    eco_path   = f'{output_dir}/{eco_id}_{safe_name}.csv'
    daily_path = f'{daily_dir}/{eco_id}_{safe_name}_daily.csv'

    # ------------------------------------------------------------------
    # CHECKPOINT — daily file is the single signal that this ecoregion
    # was fully processed. eco_path may be absent if no valid years exist.
    # ------------------------------------------------------------------
    if os.path.exists(daily_path):
        if os.path.exists(eco_path):
            existing = pd.read_csv(eco_path)
            all_metrics.extend(existing.to_dict('records'))
            print(f'  Skipping {eco_name} — already done')
        else:
            print(f'  {eco_name} — daily counts on disk but no metrics file, recomputing locally.')
            saved_daily = pd.read_csv(daily_path)
            eco_metrics = []

            for year in YEARS:
                df_year = saved_daily[saved_daily['year'] == year][['doy', 'n_detections']]
                if df_year.empty:
                    continue
                metrics = compute_timing_metrics(df_year, year)
                if metrics is not None:
                    metrics['eco_id']    = eco_id
                    metrics['eco_name']  = eco_name
                    metrics['biome_num'] = biome_num
                    metrics['biome_name']= biome_name
                    eco_metrics.append(metrics)
                    all_metrics.append(metrics)
                else:
                    failed_years.append({
                        'eco_id': eco_id, 'eco_name': eco_name,
                        'year': year, 'reason': 'metrics_none'
                    })

            n_years_valid   = len(eco_metrics)
            pct_years_valid = round(n_years_valid / len(YEARS), 3)
            for m in eco_metrics:
                m['n_years_valid']   = n_years_valid
                m['pct_years_valid'] = pct_years_valid

            if eco_metrics:
                pd.DataFrame(eco_metrics).to_csv(eco_path, index=False)
                print(f'  Recovered {len(eco_metrics)} metric years from daily file')
            else:
                print(f'  No valid fire years for {eco_name} — confirmed from daily file')

        continue

    # ------------------------------------------------------------------
    # GEE FETCH — only reaches here if daily_path does not exist
    # ------------------------------------------------------------------
    print(f'\n=== {eco_name} (ID: {eco_id}) ===')
    eco_metrics   = []
    daily_records = []

    for year in YEARS:
        t0 = time.time()

        try:
            df_year = get_daily_counts(geometry, year)

            df_year['year']     = year
            df_year['eco_id']   = eco_id
            df_year['eco_name'] = eco_name
            daily_records.extend(df_year.to_dict('records'))

            metrics = compute_timing_metrics(df_year, year)

            if metrics is not None:
                metrics['eco_id']    = eco_id
                metrics['eco_name']  = eco_name
                metrics['biome_num'] = biome_num
                metrics['biome_name']= biome_name
                eco_metrics.append(metrics)
                all_metrics.append(metrics)
            else:
                failed_years.append({
                    'eco_id': eco_id, 'eco_name': eco_name,
                    'year': year, 'reason': 'metrics_none'
                })

        except Exception as e:
            failed_years.append({
                'eco_id': eco_id, 'eco_name': eco_name,
                'year': year, 'reason': f'exception: {e}'
            })
            print(f'  {year}: ERROR — {e}')
            continue

        t1 = time.time()
        print(f'  {year}: done in {t1 - t0:.1f}s')

    # Save daily counts (always, even if all metric years failed)
    if daily_records:
        daily_df = pd.DataFrame(daily_records)[
            ['eco_id', 'eco_name', 'year', 'doy', 'n_detections']
        ]
        daily_df.to_csv(daily_path, index=False)
        print(f'  Saved {len(daily_df)} daily rows → {os.path.abspath(daily_path)}')

    # Quality flags across all valid years for this ecoregion
    n_years_valid   = len(eco_metrics)
    pct_years_valid = round(n_years_valid / len(YEARS), 3)
    for m in eco_metrics:
        m['n_years_valid']   = n_years_valid
        m['pct_years_valid'] = pct_years_valid

    # Save metrics CSV
    if eco_metrics:
        eco_df = pd.DataFrame(eco_metrics)
        eco_df.to_csv(eco_path, index=False)
        print(f'  Saved {len(eco_metrics)} metric years → {os.path.abspath(eco_path)}')
    else:
        print(f'  No valid metric years for {eco_name}.')

    # Update failed log after each ecoregion
    if failed_years:
        pd.DataFrame(failed_years).to_csv(f'{output_dir}/_failed.csv', index=False)

print('\nAll ecoregions complete.')

Ecoregions:   0%|          | 0/14 [00:00<?, ?it/s]


=== Anatolian conifer and deciduous mixed forests (ID: 786) ===
  2003: done in 1.6s
  2004: done in 1.7s
  2005: done in 1.5s
  2006: done in 2.1s
  2007: done in 1.7s
  2008: done in 2.1s
  2009: done in 1.4s
  2010: done in 2.3s
  2011: done in 1.4s
  2012: done in 1.9s
  2013: done in 2.4s
  2014: done in 2.1s
  2015: done in 2.3s
  2016: done in 2.3s
  2017: done in 1.4s
  2018: done in 2.1s
  2019: done in 1.9s
  2020: done in 1.6s
  2021: done in 1.7s
  2022: done in 1.8s
  2023: done in 2.2s
  2024: done in 2.0s


Ecoregions:   7%|▋         | 1/14 [00:43<09:28, 43.75s/it]

  2025: done in 2.3s
  Saved 8401 daily rows → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\daily_counts\786_Anatolian_conifer_and_deciduous_mixed_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\786_Anatolian_conifer_and_deciduous_mixed_forests.csv

=== Southern Anatolian montane conifer and deciduous forests (ID: 804) ===
  2003: done in 1.7s
  2004: done in 1.4s
  2005: done in 2.0s
  2006: done in 1.9s
  2007: done in 2.0s
  2008: done in 1.9s
  2009: done in 2.0s
  2010: done in 1.5s
  2011: done in 1.8s
  2012: done in 2.1s
  2013: done in 2.6s
  2014: done in 2.0s
  2015: done in 2.3s
  2016: done in 1.8s
  2017: done in 2.0s
  2018: done in 2.7s
  2019: done in 1.9s
  2020: done in 1.8s
  2021: done in 2.5s
  2022: done in 8.4s
  2023: done in 2.1s
  2024: done in 2.0s


Ecoregions:  14%|█▍        | 2/14 [01:36<09:46, 48.87s/it]

...

  2025: done in 1.9s
  Saved 8401 daily rows → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\daily_counts\739_Syrian_xeric_grasslands_and_shrublands_daily.csv
  Saved 23 metric years → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\739_Syrian_xeric_grasslands_and_shrublands.csv

=== Aegean and Western Turkey sclerophyllous and mixed forests (ID: 785) ===
  2003: done in 4.6s
  2004: done in 5.4s
  2005: done in 4.9s
  2006: done in 5.6s
  2007: done in 4.5s
  2008: done in 4.3s
  2009: done in 4.2s
  2010: done in 5.7s
  2011: done in 5.2s
  2012: done in 5.2s
  2013: done in 5.0s
  2014: done in 6.4s
  2015: done in 4.6s
  2016: done in 5.1s
  2017: done in 5.5s
  2018: done in 4.4s
  2019: done in 5.5s
  2020: done in 4.9s
  2021: done in 4.2s
  2022: done in 5.1s
  2023: done in 4.2s
  2024: done in 6.4s


Ecoregions:  93%|█████████▎| 13/14 [12:13<01:09, 69.61s/it]

  2025: done in 4.5s
  Saved 8401 daily rows → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\daily_counts\785_Aegean_and_Western_Turkey_sclerophyllous_and_mixed_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\785_Aegean_and_Western_Turkey_sclerophyllous_and_mixed_forests.csv

=== Eastern Mediterranean conifer-broadleaf forests (ID: 791) ===
  2003: done in 5.7s
  2004: done in 12.5s
  2005: done in 5.8s
  2006: done in 12.0s
  2007: done in 7.2s
  2008: done in 6.9s
  2009: done in 5.9s
  2010: done in 9.3s
  2011: done in 6.1s
  2012: done in 7.2s
  2013: done in 9.6s
  2014: done in 7.3s
  2015: done in 6.1s
  2016: done in 7.5s
  2017: done in 6.9s
  2018: done in 6.5s
  2019: done in 5.8s
  2020: done in 14.4s
  2021: done in 5.6s
  2022: done in 6.2s
  2023: done in 6.0s
  2024: done in 6.8s


Ecoregions: 100%|██████████| 14/14 [15:06<00:00, 64.76s/it] 

  2025: done in 5.8s
  Saved 8401 daily rows → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\daily_counts\791_Eastern_Mediterranean_conifer-broadleaf_forests_daily.csv
  Saved 23 metric years → C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\791_Eastern_Mediterranean_conifer-broadleaf_forests.csv

All ecoregions complete.


In [20]:
# POST-RUN ASSEMBLY — METRICS AND DAILY COUNTS -----------------------------------------------------

import glob

# Assemble metrics
metric_files = sorted(glob.glob(f'{output_dir}/[!_]*.csv'))
if metric_files:
    metrics_combined = pd.concat(
        [pd.read_csv(f) for f in metric_files], ignore_index=True
    )
    metrics_combined.to_csv(f'{output_dir}/_all_metrics.csv', index=False)
    print(f'Metrics:      {len(metric_files)} files → {len(metrics_combined)} rows')

# Assemble daily counts
daily_files = sorted(glob.glob(f'{daily_dir}/[!_]*_daily.csv'))
if daily_files:
    daily_combined = pd.concat(
        [pd.read_csv(f) for f in daily_files], ignore_index=True
    )
    daily_combined.to_csv(f'{output_dir}/_all_daily_counts.csv', index=False)
    print(f'Daily counts: {len(daily_files)} files → {len(daily_combined)} rows')

Metrics:      14 files → 304 rows
Daily counts: 14 files → 117614 rows


In [21]:
# COMBINE ALL RESULTS INTO MASTER CSV --------------------------------------------------------------

master_df = pd.DataFrame(all_metrics)

master_df = master_df[[
    'eco_id', 'eco_name', 'biome_num', 'biome_name',
    'year', 'onset_doy', 'peak_doy', 'end_doy', 'season_length',
    'n_detections', 'onset_month', 'peak_month',
    'peak_outside_window', 'n_years_valid', 'pct_years_valid'
]]

master_path = f'{output_dir}/master_turkey.csv'
master_df.to_csv(master_path, index=False)

print(f'Master CSV saved: {master_df.shape[0]} ecoregion-year rows.')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10))

Master CSV saved: 304 ecoregion-year rows.
Path: C:\Users\ibekar\My Drive\fire-seasons\turkey_ecoregions\master_turkey.csv

   eco_id                                       eco_name  biome_num  \
0     786  Anatolian conifer and deciduous mixed forests         12   
1     786  Anatolian conifer and deciduous mixed forests         12   
2     786  Anatolian conifer and deciduous mixed forests         12   
3     786  Anatolian conifer and deciduous mixed forests         12   
4     786  Anatolian conifer and deciduous mixed forests         12   
5     786  Anatolian conifer and deciduous mixed forests         12   
6     786  Anatolian conifer and deciduous mixed forests         12   
7     786  Anatolian conifer and deciduous mixed forests         12   
8     786  Anatolian conifer and deciduous mixed forests         12   
9     786  Anatolian conifer and deciduous mixed forests         12   

                                 biome_name  year  onset_doy  peak_doy  \
0  Mediterranean For